<a id="top"></a>

# Demo: the same graph, driving a real model (+ the one-line preview)

```yaml
title: "Demo: the same graph, driving a real model (+ the one-line preview)"
keywords: ReAct agent, StateGraph, live model, langchain, init_chat_model, ChatAnthropic, stream updates, multi-step, natural termination, create_agent, tools_condition, behavioral equivalence, sonnet, teacher demo
estimated duration: 15
```

> **Lesson:** L10. The live counterpart to [L1003](L1003_lecture.ipynb). Makes
> **live** Claude calls when `ANTHROPIC_API_KEY` is set (copy `.env.example` to `.env`); with no key
> it prints skip lines so a keyless restart-and-run-all still passes for you. A real model **varies**:
> dry-run it before class. **Clear outputs before committing.**
>
> Two things to take away here: (1) the graph you built on a stub is the **same graph** that drives a
> real model — only the model object changes; (2) `create_agent` (next lesson) builds **this exact
> graph** in one line. For now, just look — the one-liner's knobs are L11's job.

## Contents

- [1. Setup — a real model and the tools](#1-setup--a-real-model-and-the-tools)
- [2. The same graph, unchanged](#2-the-same-graph-unchanged)
- [3. Run it live — a task that forces a chain](#3-run-it-live--a-task-that-forces-a-chain)
- [4. The graph *is* the loop; the prebuilt is one line](#4-the-graph-is-the-loop-the-prebuilt-is-one-line)
- [5. Takeaways](#5-takeaways)

## 1. Setup — a real model and the tools

A **LangChain chat model** (`init_chat_model("anthropic:…")`, key via the config seam) and the same
`calculator` / `lookup` / `flaky_fetch` tools. Note what's **gone**: hand-written JSON tool schemas.
Binding infers each tool's schema from its **type hints and docstring**, so the typed function *is*
the schema — which is why these carry precise signatures and one-line docstrings. Live calls are
gated on `LIVE`, so a keyless run still completes. Anchor model: **Claude Sonnet 4.6**.

In [1]:
from typing import Annotated, Any, TypedDict

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common.config import get_settings

MODEL = "anthropic:claude-sonnet-4-6"
LIVE = bool(get_settings().anthropic_api_key)


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the exact result.

    Use for any arithmetic, e.g. '17*17 - 1'. Raises ValueError on non-arithmetic input.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the whitelist above blocks names/attributes/calls.
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Look up a city's population from a small internal table.

    Takes a city name, e.g. 'Tokyo'. Raises KeyError if the city is not on file.
    """
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


def flaky_fetch(url: str) -> str:
    """Fetch a URL from a tiny fake network. https://crash raises; other urls return a body."""
    if url == "https://crash":
        raise RuntimeError("connection reset by peer")
    return "the answer is 42"


TOOLS = [calculator, lookup, flaky_fetch]
print("live model:", MODEL if LIVE else "(no ANTHROPIC_API_KEY set — live cells will skip)")

live model: (no ANTHROPIC_API_KEY set — live cells will skip)


[↑ Back to top](#top)

## 2. The same graph, unchanged

This `build_agent` is **byte-for-byte** the graph from the [L1003 stub demo](L1003_lecture.ipynb):
`agent` node, prebuilt `ToolNode`, a `route` conditional edge, and the `tools → agent` back-edge.
Because the graph only ever touches `model.bind_tools(...)` / an awaited `.ainvoke(...)`, a real
`ChatAnthropic` and the `FakeModel` are interchangeable — swap the model object, keep the graph.

In [2]:
class AgentState(TypedDict):
    """add_messages APPENDS each node's messages to the running conversation."""

    messages: Annotated[list[BaseMessage], add_messages]


def build_agent(model: Any) -> Any:
    """The L1003 graph, unchanged: agent node + ToolNode + conditional route + back-edge."""

    async def agent_node(state: AgentState) -> dict[str, list[BaseMessage]]:
        return {"messages": [await model.bind_tools(TOOLS).ainvoke(state["messages"])]}

    def route(state: AgentState) -> str:
        last = state["messages"][-1]
        return "tools" if (isinstance(last, AIMessage) and last.tool_calls) else END

    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(TOOLS))
    builder.set_entry_point("agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()


def describe(msg: BaseMessage) -> str:
    """One readable line per message: type, tool calls, tool results, or final text."""
    kind = type(msg).__name__
    if isinstance(msg, AIMessage) and msg.tool_calls:
        calls = ", ".join(f"{c['name']}({c['args']})" for c in msg.tool_calls)
        return f"{kind:12} -> tool call: {calls}"
    if isinstance(msg, ToolMessage):
        return f"{kind:12} <- result [{msg.status}]: {msg.content!r}"
    return f"{kind:12}    {str(msg.content)!r}"

[↑ Back to top](#top)

## 3. Run it live — a task that forces a chain

A prompt designed to force a **chain**: compute something with `calculator`, then use the result to
`lookup` a city's population, then answer. Watch it with `async for` over
`astream(stream_mode="updates")` so you can see the cycle turn — `agent` → `tools` → `agent` → … —
on a real model. A live model *varies*: the exact tool order and turn count differ run to run; the
graph's job is to be correct regardless.

In [3]:
task = (
    "Use the calculator to compute 17*17 - 1, then look up the population of Tokyo, "
    "and tell me both numbers. Use the tools; do not answer from memory."
)

if LIVE:
    live_model = init_chat_model(MODEL, api_key=get_settings().anthropic_api_key)
    graph = build_agent(live_model)
    async for chunk in graph.astream(
        {"messages": [HumanMessage(content=task)]}, stream_mode="updates"
    ):
        for node_name, update in chunk.items():
            for msg in update["messages"]:
                print(f"[node: {node_name:5}] {describe(msg)}")
else:
    print("No ANTHROPIC_API_KEY set — skipping the live run.")
    print("Set it in your environment or repo-root .env to drive Sonnet 4.6.")

No ANTHROPIC_API_KEY set — skipping the live run.
Set it in your environment or repo-root .env to drive Sonnet 4.6.


[↑ Back to top](#top)

## 4. The graph *is* the loop; the prebuilt is one line

Everything you wired by hand — `agent` node, `ToolNode`, `route`, back-edge — LangGraph can hand you
in **one line**:

```python
agent = create_agent(model, TOOLS)
```

`create_agent` builds *the exact graph you just drew*. To see that without spending a token, build
one on the offline `FakeModel` and read its structure: same two real nodes, same back-edge. The only
cosmetic difference is the model-call node is named **`model`** (you named yours **`agent`**), and
its routing is the prebuilt **`tools_condition`** — the `route` you wrote by hand.

> **Just look for now** — `create_agent`'s knobs (system prompt, structured output, error handling)
> belong to **[L11](../L11/L1101_intro.md)**. All you need here: the one line builds this graph.

In [4]:
from fluffy_potato_curriculum.common.fake_model import FakeModel, text_reply

# Build both graphs on a throwaway stub (no key needed) and compare their structure.
hand_wired = build_agent(FakeModel([text_reply("x")])).get_graph()
prebuilt = create_agent(FakeModel([text_reply("x")]), TOOLS).get_graph()


def shape(graph: Any) -> tuple[list[str], list[tuple[str, str]]]:
    """The real (non-start/end) nodes and edges of a compiled graph."""
    nodes = [n for n in graph.nodes if not n.startswith("__")]
    edges = [(e.source, e.target) for e in graph.edges]
    return nodes, edges


hand_nodes, hand_edges = shape(hand_wired)
pre_nodes, pre_edges = shape(prebuilt)
print("hand-wired nodes:", hand_nodes, " back-edge:", ("tools", "agent") in hand_edges)
print("create_agent nodes:", pre_nodes, " back-edge:", ("tools", "model") in pre_edges)
print(
    "\nSame shape: two nodes (model-call + tools) and a back-edge — one built by hand, one by create_agent."
)

hand-wired nodes: ['agent', 'tools']  back-edge: True
create_agent nodes: ['model', 'tools']  back-edge: True

Same shape: two nodes (model-call + tools) and a back-edge — one built by hand, one by create_agent.


And live, the two are behaviorally equivalent — the same chaining task, the same answer, from a
hand-wired graph and from the one-liner. (Gated on a key.)

In [5]:
if LIVE:
    live_model = init_chat_model(MODEL, api_key=get_settings().anthropic_api_key)

    hand_result = await build_agent(live_model).ainvoke({"messages": [HumanMessage(content=task)]})
    one_liner = create_agent(live_model, TOOLS)
    prebuilt_result = await one_liner.ainvoke({"messages": [HumanMessage(content=task)]})

    print("hand-wired final :", hand_result["messages"][-1].content)
    print("create_agent final:", prebuilt_result["messages"][-1].content)
else:
    print("No ANTHROPIC_API_KEY set — skipping the live equivalence run.")

No ANTHROPIC_API_KEY set — skipping the live equivalence run.


[↑ Back to top](#top)

## 5. Takeaways

- **The graph is the same code** as the stub demo — only the model object changed (a real
  `ChatAnthropic` in place of the `FakeModel`). That interchangeability is why you wired the control
  flow offline first.
- **A real model varies**: the exact tool order and turn count differ run to run. The graph's job is
  to be correct regardless — `recursion_limit` exists precisely because you can't predict the model.
- **`astream(stream_mode="updates")` is the minimum-viable trace** — one chunk per node. [L12](../L12/objectives.md)
  routes *this exact run* to a structured tracer (each node a span). Keep this graph accessible.
- **`create_agent` is this graph, packaged.** Same two nodes, same back-edge — built by hand here,
  in one line next lesson. [L11](../L11/L1101_intro.md) teaches the one-liner and its knobs; you now
  have the picture of what it wraps.

[↑ Back to top](#top)